# Qwen2.5-Omni Speech-to-Speech server (free Kaggle GPU)

Hosts the ONE-model speech-to-speech Qwen model on a free Kaggle GPU so your PC runs no heavy AI.

> **Correct model ID:** `Qwen/Qwen2.5-Omni-3B` (there is **NO `-Instruct`** suffix — that ID
> does not exist and shows a 404 / "Repository Not Found").

> **Gated models:** if the load cell reports a 401/403, set a `HF_TOKEN` secret (Settings →
> Secrets) after accepting the license on the model page.

**Important:** Qwen has NO 1B/2B speech-to-speech model. The smallest native voice-in->voice-out model is **Qwen2.5-Omni-3B**. We load it in 4-bit so it fits a 16GB Kaggle T4 GPU.


## Setup
1. Kaggle Settings → **Accelerator: GPU T4 x2 (16GB)**.
2. Model ID is **`Qwen/Qwen2.5-Omni-3B`** (no `-Instruct`).
3. **Only if the load cell gives 401/403:** accept the license at
   https://huggingface.co/Qwen/Qwen2.5-Omni-3B , create a read token at
   https://huggingface.co/settings/tokens , then add it as a Kaggle **Secret** named `HF_TOKEN`.
4. Run this cell to install deps.


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece
!pip -q install torchaudio soundfile librosa fastapi uvicorn python-multipart pyngrok
import transformers, torch
print('transformers', transformers.__version__)   # must be a recent 4.x


## Load the model (Qwen2.5-Omni-3B, 4-bit)

In [ ]:
import os, torch, gc
os.environ['HF_HOME'] = '/kaggle/working/hf'
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
MODEL_ID = 'Qwen/Qwen2.5-Omni-3B'   # correct ID — no '-Instruct'
hf_token = os.environ.get('HF_TOKEN', '').strip() or None   # only if gated

# Load ONCE. If you re-run this cell, it reuses the already-loaded model.
if 'model' in globals():
    print('model already loaded — skipping reload')
else:
    from transformers import AutoConfig, Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor, BitsAndBytesConfig

    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token)
    for _sub in ('talker_config', 'thinker_config', 'text_config'):
        _sc = getattr(config, _sub, None)
        if _sc is not None and not hasattr(_sc, 'pad_token_id'):
            _sc.pad_token_id = getattr(config, 'pad_token_id', 151643)
            print('patched', _sub, '.pad_token_id')

    # 4-bit quantization is ESSENTIAL: bf16 of this ~6B model uses ~12GB, which
    # leaves no room for the audio-codec (DiT) generation -> CUDA out of memory.
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
        MODEL_ID, config=config, quantization_config=bnb, device_map='auto',
        trust_remote_code=True, token=hf_token)
    processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token)
    print('model loaded on:', model.device)

import torch as _t
def _free():
    _t.cuda.empty_cache(); gc.collect()


## Start the FastAPI server + public tunnel
The printed `STS: https://.../sts` URL goes into your `config/config.yaml` → `sts.qwen_kaggle.url`.

In [ ]:
import base64, io, json, threading
import torch
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse
import uvicorn

def gen_kwargs():
    return {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
            'do_sample': True, 'max_new_tokens': 160}

app = FastAPI(title='Qwen Omni STS')

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_ID}

@app.post('/sts')
async def sts(file: UploadFile = File(...), system: str = Form(''), history: str = Form('[]')):
    import librosa
    raw = await file.read()
    audio, sr = librosa.load(io.BytesIO(raw), sr=16000, mono=True)
    try:
        hist = json.loads(history or '[]')
    except Exception:
        hist = []
    user_prompt = ('This is a phone conversation. Reply aloud in the SAME language '
                   'the speaker used. Answer from the provided context only. '
                   'Keep it short and natural.')
    messages = [{'role': 'system', 'content': system}]
    for h in hist:
        role = 'assistant' if h.get('role') == 'assistant' else 'user'
        messages.append({'role': role, 'content': h.get('text', '')})
    messages.append({'role': 'user', 'content': [
        {'type': 'audio', 'audio_url': 'uploaded.wav'},
        {'type': 'text', 'text': user_prompt}]})
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, audio=[audio], return_tensors='pt', padding=True)
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    _free()  # reclaim GPU memory before generation
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs(), return_audio=True)
    _free()
    reply = out['text']
    reply = reply[0] if isinstance(reply, list) else reply
    reply = reply.split('<|im_end|>')[0].strip()
    audio_b64, sr_out = '', 24000
    if out.get('audio') is not None:
        wav = out['audio'][0].float().cpu().numpy()
        sr_out = out.get('sampling_rate')
        audio_b64 = base64.b64encode(wav.tobytes()).decode()
    return JSONResponse({'text': reply, 'audio_b64': audio_b64,
                         'sample_rate': int(sr_out), 'language': 'auto'})

# Free the port if a previous run's server is still alive.
port = 8501
import socket
s = socket.socket();
try:
    s.bind(('0.0.0.0', port)); s.close()   # port free
except OSError:
    print('Port %d in use -> RESTART the kernel (Kernel -> Restart & Run All) to free it.' % port)
cfg = uvicorn.Config(app, host='0.0.0.0', port=port, log_level='info')
server = uvicorn.Server(cfg)
threading.Thread(target=server.run, daemon=True).start()


from pyngrok import ngrok

# --- ngrok tunnel needs YOUR authtoken (free account: https://ngrok.com) ---
# Option A (recommended): add a Kaggle Secret named NGROK_AUTHTOKEN
# Option B: paste your token directly inside the quotes below.
NGROK_TOKEN = os.environ.get('NGROK_AUTHTOKEN', '').strip()
if not NGROK_TOKEN:
    NGROK_TOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'
if NGROK_TOKEN and 'PASTE_' not in NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    raise SystemExit('Set your ngrok authtoken: add a Kaggle Secret named '
                     'NGROK_AUTHTOKEN, or paste it in this cell.')

url = ngrok.connect(port, bind_tls=True).public_url
print('========== QWEN OMNI STS READY ==========')
print('HEALTH:', url + '/health')
print('STS   :', url + '/sts')
print('Copy the STS URL into config sts.qwen_kaggle.url')
print('=========================================')

## Done
Copy the `STS: https://.../sts` URL into `config/config.yaml` → `sts.qwen_kaggle.url`, then on your PC run:
```bash
python main.py --sts --audio your_voice.wav
```